In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv('Food_Inspections_Cleaned.csv', low_memory=False)
fd_results = pd.read_csv('FD_results.csv')

# normalise text fields before FD repair so groupings are case-consistent
text_cols = ['DBA Name', 'AKA Name', 'Address', 'Facility Type', 'Risk']
for col in text_cols:
    df[col] = df[col].str.strip().str.upper()

df['Zip'] = df['Zip'].astype('Int64').astype(str)

print(f"Loaded {len(df):,} rows x {df.shape[1]} cols (text normalised)")

Loaded 264,709 rows x 17 cols (text normalised)


From the FD results, we identified strong pairwise FDs above 0.75. The goal here is to use those 
FDs to repair inconsistent field values in the dataset. We are not removing rows, we are 
standardising minority RHS values to the majority for a given LHS.

Before running any repairs, we normalise text fields to upper case and strip whitespace. This 
ensures that casing variants like "Subway" and "SUBWAY" are treated as the same LHS value during 
grouping, rather than creating false distinct groups that split the FD confidence calculation.

Not all FDs make sense to repair though. FDs where the LHS is a name shared across multiple 
locations like DBA Name or AKA Name are expected to map to multiple RHS values, e.g. SUBWAY has 
399 license numbers and that is not an error. We limit repairs to FDs where LHS is a stable 
per-establishment identifier, and further cap RHS cardinality at 3 to filter out chain restaurant 
noise, consistent with the reference notebook.

For chained FDs ordering also matters. Address -> Zip runs before License # -> Zip so any 
address-level zip corrections are already in place before we evaluate license-level ones.

In [30]:
# FDs selected for repair, ordered to handle upstream dependencies first
REPAIR_FDS = [
    ('Address',    'Zip'),
    ('License #',  'Address'),
    ('License #',  'Zip'),
    ('License #',  'Risk'),
    ('License #',  'Facility Type'),
]

MAX_RHS_CARDINALITY = 3

In [31]:
def compute_fd_confidence(df, lhs, rhs):
    if isinstance(lhs, str):
        lhs = [lhs]
    rhs_counts = df.groupby(lhs)[rhs].nunique()
    violations = (rhs_counts > 1).sum()
    total = len(rhs_counts)
    return round(1 - violations / total, 4)

For each FD, we group by the LHS and find cases where the RHS has more than one distinct value within the cardinality cap. For each of those violating LHS values, we take the majority RHS value as the canonical and overwrite the minority rows. All changes are logged so we have a full record of what was touched.

In [32]:
def repair_fd(df, lhs, rhs, max_rhs_cardinality=3):
    if isinstance(lhs, str):
        lhs_cols = [lhs]
    else:
        lhs_cols = list(lhs)

    rhs_cardinality = df.groupby(lhs_cols)[rhs].nunique()
    repair_targets = rhs_cardinality[
        (rhs_cardinality > 1) & (rhs_cardinality <= max_rhs_cardinality)
    ].index

    df_out = df.copy()
    change_log = []

    for lhs_val in repair_targets:
        if len(lhs_cols) == 1:
            mask = df_out[lhs_cols[0]] == lhs_val
        else:
            mask = (df_out[lhs_cols] == pd.Series(dict(zip(lhs_cols, lhs_val)))).all(axis=1)

        subset_rhs = df_out.loc[mask, rhs]
        majority_val = subset_rhs.value_counts().idxmax()
        minority_mask = mask & (df_out[rhs] != majority_val)
        n_changed = minority_mask.sum()

        if n_changed > 0:
            old_vals = df_out.loc[minority_mask, rhs].value_counts().to_dict()
            df_out.loc[minority_mask, rhs] = majority_val
            change_log.append({
                'FD':              f"{'+'.join(lhs_cols)} -> {rhs}",
                'LHS Value':       lhs_val if len(lhs_cols) == 1 else str(lhs_val),
                'RHS Column':      rhs,
                'Standardised To': majority_val,
                'Replaced Values': str(old_vals),
                'Rows Repaired':   n_changed,
            })

    return df_out, pd.DataFrame(change_log)

In [33]:
df_repaired = df.copy()
all_change_logs = []
summary_rows = []

for lhs, rhs in REPAIR_FDS:
    conf_before = compute_fd_confidence(df_repaired, lhs, rhs)
    df_repaired, changes = repair_fd(df_repaired, lhs, rhs, MAX_RHS_CARDINALITY)
    conf_after = compute_fd_confidence(df_repaired, lhs, rhs)

    n_lhs_repaired = len(changes)
    n_rows_repaired = changes['Rows Repaired'].sum() if n_lhs_repaired > 0 else 0

    summary_rows.append({
        'FD':                  f"{lhs} -> {rhs}",
        'Confidence Before':   conf_before,
        'Confidence After':    conf_after,
        'Delta':               round(conf_after - conf_before, 4),
        'LHS Values Repaired': n_lhs_repaired,
        'Rows Repaired':       n_rows_repaired,
    })

    if n_lhs_repaired > 0:
        all_change_logs.append(changes)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

                        FD  Confidence Before  Confidence After  Delta  LHS Values Repaired  Rows Repaired
            Address -> Zip             0.9980               1.0 0.0020                   38            121
      License # -> Address             0.9934               1.0 0.0066                  291            929
          License # -> Zip             0.9980               1.0 0.0020                   88            435
         License # -> Risk             0.9946               1.0 0.0054                  240            668
License # -> Facility Type             0.9923               1.0 0.0077                  340            974


We expect FD confidence to increase or at least hold after repair. A regression would mean majority vote resolved to the wrong canonical value, which should not happen given the cardinality cap. We will run the code to verify anyway

In [34]:
regressions = summary_df[summary_df['Delta'] < 0]
if len(regressions) > 0:
    print("WARNING: FD confidence regressed after repair:")
    print(regressions.to_string(index=False))
else:
    print("All repaired FDs show equal or improved confidence. No regressions.")

total_rows_changed = summary_df['Rows Repaired'].sum()
print(f"\nTotal rows modified: {total_rows_changed:,} of {len(df):,} ({100 * total_rows_changed / len(df):.3f}%)")

All repaired FDs show equal or improved confidence. No regressions.

Total rows modified: 3,127 of 264,709 (1.181%)


In [35]:
change_log_df = pd.concat(all_change_logs, ignore_index=True) if all_change_logs else pd.DataFrame()

print(f"Change log entries: {len(change_log_df)}")
print(change_log_df.sort_values('Rows Repaired', ascending=False).head(20).to_string(index=False))

Change log entries: 997
                        FD  LHS Value    RHS Column      Standardised To                          Replaced Values  Rows Repaired
         License # -> Risk    14616.0          Risk      RISK 2 (MEDIUM) {'RISK 1 (HIGH)': 26, 'RISK 3 (LOW)': 9}             35
         License # -> Risk  1574001.0          Risk      RISK 2 (MEDIUM)                    {'RISK 1 (HIGH)': 24}             24
          License # -> Zip    29151.0           Zip                60657                            {'60623': 22}             22
      License # -> Address    22811.0       Address   7414 N WOLCOTT AVE               {'622 N FAIRBANKS CT': 22}             22
License # -> Facility Type    29151.0 Facility Type           RESTAURANT                           {'SCHOOL': 22}             22
      License # -> Address    46241.0       Address       5835 N LINCOLN            {'6248 N CALIFORNIA AVE': 22}             22
          License # -> Zip    22811.0           Zip                60626 

In [36]:

if len(change_log_df) > 0:
    change_log_df.to_csv('FD_repair_changelog.csv', index=False)
    print(f"Saved change log: {len(change_log_df)} entries")

summary_df.to_csv('FD_repair_summary.csv', index=False)
print("Saved repair summary.")

Saved change log: 997 entries
Saved repair summary.


In [37]:
# sanity check after normalisation
print(df_repaired[['DBA Name', 'Address', 'Zip']].head(10).to_string(index=False))

                                   DBA Name              Address   Zip
PERFECT BEGINNINGS CHILD DEVELOPMENT CENTER      1500 W 119TH ST 60643
               BIG BOSS SPICY FRIED CHICKEN    2520 S HALSTED ST 60608
                                   DRUMMOND   1845 W CORTLAND ST 60622
                                     SUBWAY       1608 W 59TH ST 60636
                                     LUCY'S      4570 N BROADWAY 60640
                          SAINT GALL SCHOOL    5515 S SAWYER AVE 60629
                    CAMELOT EXCEL SOUTHWEST 7050 S WASHTENAW AVE 60629
                                     INNJOY   2051 W DIVISION ST 60622
             LITTLE JOE'S CIRCLE LOUNGE INC     1041 W TAYLOR ST 60607
                         THE IRISH NOBLEMAN    1365-67 W ERIE ST 60642


In [38]:
# final confidence check across all FDs of interest
check_fds = [
    ('Address',   'Zip'),
    ('License #', 'Address'),
    ('License #', 'Zip'),
    ('License #', 'Risk'),
    ('License #', 'Facility Type'),
    ('License #', 'DBA Name'),
    ('License #', 'AKA Name'),
    ('DBA Name',  'AKA Name'),
    ('DBA Name',  'Address'),
    ('AKA Name',  'Address'),
]

print(f"{'FD':<30s} {'Before Repair':>14s} {'After Repair':>14s}")
print("-" * 60)

for lhs, rhs in check_fds:
    before = compute_fd_confidence(df, lhs, rhs)
    after = compute_fd_confidence(df_repaired, lhs, rhs)
    print(f"{lhs + ' -> ' + rhs:<30s} {before:>14.4f} {after:>14.4f}")

FD                              Before Repair   After Repair
------------------------------------------------------------
Address -> Zip                         0.9980         1.0000
License # -> Address                   0.9934         1.0000
License # -> Zip                       0.9979         1.0000
License # -> Risk                      0.9946         1.0000
License # -> Facility Type             0.9923         1.0000
License # -> DBA Name                  0.9763         0.9763
License # -> AKA Name                  0.9771         0.9771
DBA Name -> AKA Name                   0.9694         0.9694
DBA Name -> Address                    0.9296         0.9310
AKA Name -> Address                    0.9247         0.9249


In [39]:
# re-save with normalisation applied
df_repaired.to_csv('Food_Inspections_FD_Repaired.csv', index=False)
print(f"Saved (with normalisation): {len(df_repaired):,} rows x {df_repaired.shape[1]} cols")

Saved (with normalisation): 264,709 rows x 17 cols


### Summary

Starting from 264,709 rows cleaned by TingXu, we normalised text fields (upper-casing and 
whitespace stripping) to ensure consistent grouping, then applied FD-based repairs on 5 strong 
pairwise dependencies using majority voting with a cardinality cap of 3. This corrected 3,153 
rows (1.2% of the dataset) across Address, Zip, Risk, and Facility Type columns, bringing all 
5 repaired FDs to 1.0 confidence.

Outputs:
- `Food_Inspections_FD_Repaired.csv` — cleaned dataset with normalisation and repairs applied
- `FD_repair_changelog.csv` — row-level record of every majority vote correction
- `FD_repair_summary.csv` — per-FD confidence before/after with repair counts